# Construcción de la tabla de dirección

En este notebook leo los créditos descargados y construyo la tabla de
personas que dirigen cada película. Con ella creo la variable
`grupo_direccion`, que clasifica cada película según el género de las
personas acreditadas en la dirección.

Utilizo el código de género que informa TMDB. Ese dato no es una
autoidentificación verificada y se tratará como tal en las limitaciones.

## 1. Lectura de los créditos y extracción de la dirección

Recorro los 681.836 archivos de créditos y me quedo con las personas
cuyo puesto es `Director`. Una película puede tener varias personas
dirigiendo, por lo que conservo todas, sin duplicar la misma persona
en una misma película. No incluyo otros puestos como `Co-Director`;
más abajo compruebo que no se queda fuera a nadie por este motivo.

Leer tantos archivos uno a uno es lento, por lo que la lectura se hace
con 16 hilos y en 14 tramos de 50.000 archivos. Cada tramo se guarda en
`tramos_direccion/` al terminar. Si el proceso se interrumpe, al volver
a ejecutar la celda se saltan los tramos ya guardados.

La salida que aparece debajo corresponde a la primera ejecución, que
tardó alrededor de un minuto por tramo. No hace falta repetirla.

In [1]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import pandas as pd
from dotenv import find_dotenv

raiz = Path(find_dotenv(usecwd=True)).resolve().parent
carpeta_creditos = raiz / "src" / "data" / "raw" / "creditos"
carpeta_proc = raiz / "src" / "data" / "processed"
carpeta_tramos = carpeta_proc / "tramos_direccion"
carpeta_tramos.mkdir(parents=True, exist_ok=True)

TAMANO_TRAMO = 50_000
HILOS = 16

def leer(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        datos = json.load(f)
    filas = [
        {"id_pelicula": datos["id"], "id_persona": p["id"],
         "nombre": p["name"], "gender": p["gender"]}
        for p in datos["crew"] if p.get("job") == "Director"
    ]
    return datos["id"], filas

rutas = sorted(carpeta_creditos.glob("creditos_*.json"))
total = len(rutas)
n_tramos = (total + TAMANO_TRAMO - 1) // TAMANO_TRAMO
print("Archivos de créditos:", total, "| Tramos:", n_tramos)

with ThreadPoolExecutor(max_workers=HILOS) as ejecutor:
    for num in range(n_tramos):
        destino = carpeta_tramos / f"tramo_{num:03d}.pkl"
        if destino.exists():
            print(f"Tramo {num + 1}/{n_tramos}: ya guardado, se salta")
            continue

        lote = rutas[num * TAMANO_TRAMO:(num + 1) * TAMANO_TRAMO]
        resultados = list(ejecutor.map(leer, lote))

        ids_tramo = [r[0] for r in resultados]
        filas_tramo = [fila for r in resultados for fila in r[1]]

        temporal = destino.with_suffix(".tmp")
        pd.to_pickle({"ids": ids_tramo, "filas": filas_tramo}, temporal)
        temporal.replace(destino)
        print(f"Tramo {num + 1}/{n_tramos} guardado ({len(ids_tramo)} películas)", flush=True)

Archivos de créditos: 681836 | Tramos: 14
Tramo 1/14: ya guardado, se salta
Tramo 2/14: ya guardado, se salta
Tramo 3/14: ya guardado, se salta
Tramo 4/14: ya guardado, se salta
Tramo 5/14: ya guardado, se salta
Tramo 6/14: ya guardado, se salta
Tramo 7/14: ya guardado, se salta
Tramo 8/14: ya guardado, se salta
Tramo 9/14: ya guardado, se salta
Tramo 10/14: ya guardado, se salta
Tramo 11/14: ya guardado, se salta
Tramo 12/14: ya guardado, se salta
Tramo 13/14: ya guardado, se salta
Tramo 14/14: ya guardado, se salta


In [2]:
ids_con_creditos = []
filas_dir = []

for ruta_tramo in sorted(carpeta_tramos.glob("tramo_*.pkl")):
    contenido = pd.read_pickle(ruta_tramo)
    ids_con_creditos.extend(contenido["ids"])
    filas_dir.extend(contenido["filas"])

df_dir = pd.DataFrame(filas_dir).drop_duplicates(["id_pelicula", "id_persona"])

print("Películas con créditos:", len(ids_con_creditos), "(esperadas: 681836)")
print("Créditos de dirección:", len(df_dir))
display(df_dir["gender"].value_counts().sort_index())

df_dir.to_pickle(carpeta_proc / "direccion.pkl")
pd.Series(ids_con_creditos).to_pickle(carpeta_proc / "ids_con_creditos.pkl")

Películas con créditos: 681836 (esperadas: 681836)
Créditos de dirección: 690938


gender
0    370692
1     66183
2    251638
3      2425
Name: count, dtype: int64

### Comprobación del puesto «Director»

Antes de clasificar, reviso los puestos que contienen la palabra
«director» para comprobar que no estoy dejando fuera a codirectores.
Lo hago con las primeras 3.000 películas con créditos, por orden de
archivo.

In [3]:
# Puestos relacionados con dirección, en las 3.000 primeras películas con créditos
import collections
contador = collections.Counter()
for ruta in sorted(carpeta_creditos.glob("creditos_*.json"))[:3000]:
    with open(ruta, "r", encoding="utf-8") as f:
        for p in json.load(f)["crew"]:
            if "director" in p.get("job", "").lower():
                contador[p["job"]] += 1
display(pd.Series(contador).sort_values(ascending=False).head(10))

Director                     3064
Director of Photography       972
First Assistant Director      210
Assistant Director            152
Second Assistant Director     142
Casting Director               72
Third Assistant Director       59
Assistant Art Director         48
Production Director            44
Sound Director                 34
dtype: int64

En las primeras 3.000 películas con créditos, el puesto `Director`
aparece 3.064 veces. Los demás puestos que contienen la palabra
«director» son funciones técnicas, como dirección de fotografía,
ayudantes de dirección o dirección de casting, que no corresponden a
la dirección de la película. El puesto `Co-Director` no aparece entre
los diez más frecuentes. Por tanto, se mantiene el filtro exacto
`Director`.

Estas 3.000 películas no son una muestra aleatoria, por lo que la
comprobación orienta pero no demuestra que el resultado sea igual en
todo el conjunto.

## 2. Clasificación de las películas por grupo de dirección

Asigno cada película a un único grupo según los códigos de género de
sus directores:

- **Mujeres**: todas las personas tienen código 1 (mujer) o 3 (no binaria).
- **Hombres**: todas las personas tienen código 2.
- **Mixto**: hay personas de ambos grupos.
- **No identificado**: alguna persona tiene el género sin especificar.
- **Sin dirección acreditada**: hay créditos, pero ninguna persona con puesto `Director`.
- **Sin créditos descargados**: la API devolvió un 404.

Si una película tiene una persona con género sin especificar, va a
«No identificado», aunque la otra persona sea una mujer, porque no
puedo saber si el equipo es de mujeres, de hombres o mixto.

In [4]:
def clasificar(codigos):
    codigos = set(codigos)
    if 0 in codigos:
        return "No identificado"
    hay_mujer = bool(codigos & {1, 3})
    hay_hombre = 2 in codigos
    if hay_mujer and hay_hombre:
        return "Mixto"
    if hay_mujer:
        return "Mujeres"
    return "Hombres"

grupo = df_dir.groupby("id_pelicula")["gender"].apply(clasificar)

df = pd.read_pickle(carpeta_proc / "peliculas_preparadas.pkl")
df["grupo_direccion"] = df["id"].map(grupo)

# Con créditos descargados pero sin director -> sin dirección acreditada
con_creditos = df["id"].isin(ids_con_creditos)
df.loc[con_creditos & df["grupo_direccion"].isna(), "grupo_direccion"] = "Sin dirección acreditada"
# Sin créditos descargados
df.loc[~con_creditos, "grupo_direccion"] = "Sin créditos descargados"

tam = df["grupo_direccion"].value_counts()
display(pd.DataFrame({"n": tam, "%": (tam / len(df) * 100).round(2)}))

,n,%
grupo_direccion,,
No identificado,322770,47.34
Hombres,219028,32.12
Sin dirección acreditada,79385,11.64
Mujeres,56466,8.28
Mixto,4187,0.61
Sin créditos descargados,32,0.00


## 3. Transparencia sobre la categoría «Mujeres»

Como el grupo «Mujeres» incluye a las personas no binarias, cuento
cuántas películas tienen alguna persona con código 3, para que se vea
cuánto aporta ese colectivo. También cuento las películas con varias
personas dirigiendo y alguna sin género especificado.

In [5]:
# Comprobaciones para transparencia
con_no_binaria = df_dir.loc[df_dir["gender"] == 3, "id_pelicula"].nunique()
print("Películas con alguna persona no binaria (código 3):", con_no_binaria)

# Películas con varios directores y alguno sin género especificado
n_dir = df_dir.groupby("id_pelicula").size()
cod0 = df_dir[df_dir["gender"] == 0].groupby("id_pelicula").size()
mixtas_con_0 = ((n_dir > 1) & (n_dir.index.isin(cod0.index))).sum()
print("Películas con varios directores y alguno con código 0:", mixtas_con_0)

Películas con alguna persona no binaria (código 3): 2410
Películas con varios directores y alguno con código 0: 49542


2.410 películas tienen al menos una persona no binaria (código 3) en
la dirección. Algunas de ellas pueden estar clasificadas como «Mixto»
o «No identificado», pero incluso si todas estuvieran en el grupo
«Mujeres», serían como máximo el 4,3 % de sus 56.466 películas. Por
tanto, esta categoría aporta una parte pequeña del grupo.

Además, 49.542 películas tienen varios directores y alguno con género
no especificado. Van a «No identificado» por la regla de clasificación
conservadora y suponen el 15,3 % de ese grupo (322.770 películas), de
modo que la regla afecta a una minoría.

## 4. Películas descartadas

Las películas sin director acreditado y las que no tienen créditos
descargados se excluyen del análisis de dirección. Las registro en un
archivo para poder consultarlas. Siguen siendo válidas para el análisis
general de géneros, que no depende de la dirección.

In [6]:
motivos_descarte = ["Sin dirección acreditada", "Sin créditos descargados"]
df["descartada"] = df["grupo_direccion"].isin(motivos_descarte)
df_descartadas = df[df["descartada"]].copy()

print("Películas descartadas:", len(df_descartadas), f"({len(df_descartadas)/len(df):.1%})")
display(df_descartadas["grupo_direccion"].value_counts())

columnas = ["id", "title", "original_title", "anio_estreno", "original_language",
            "vote_count", "popularity", "grupo_direccion"]
df_descartadas[columnas].to_csv(
    carpeta_proc / "peliculas_sin_director.csv", index=False, encoding="utf-8-sig"
)

# Las 20 más populares, las más relevantes para revisar
display(df_descartadas.sort_values("popularity", ascending=False)[columnas].head(20))

Películas descartadas: 79417 (11.6%)


grupo_direccion
Sin dirección acreditada    79385
Sin créditos descargados       32
Name: count, dtype: int64

,id,title,original_title,anio_estreno,original_language,vote_count,popularity,grupo_direccion
537297,1135873,GGS - Ganteng-Ganteng Sange,GGS - Ganteng-Ganteng Sange,2023,id,7,8.2787,Sin dirección acreditada
96260,373234,Český Honza,Český Honza,2007,cs,0,7.1118,Sin dirección acreditada
541211,1141868,Tante Siska,Tante Siska,2023,hi,1,6.7141,Sin dirección acreditada
102468,1053703,Radio Dazed,Radio Dazed,2008,en,0,6.6503,Sin dirección acreditada
258464,614821,Shadow Nation,Shadow Nation,2015,en,0,6.1632,Sin dirección acreditada
437499,895954,모녀카센터,모녀카센터,2021,ko,1,6.0897,Sin dirección acreditada
628659,1412113,Así se hizo ‘El juego del calamar: Temporada 2',오징어 게임: 시즌2 제작 이야기,2025,ko,463,5.3745,Sin dirección acreditada
614676,1372212,22 Words,22 Words,2024,en,1,5.2592,Sin dirección acreditada
406912,669444,Les chevaliers du fiel dynamitent 2019,Les chevaliers du fiel dynamitent 2019,2020,fr,8,5.1385,Sin dirección acreditada
357811,1253611,Pygmalion,Pygmalion,2018,de,0,5.1324,Sin dirección acreditada


### ¿El descarte es aleatorio?

Si las películas descartadas fueran sobre todo películas pequeñas o de
ciertos años, el descarte podría sesgar los resultados. Comparo el
porcentaje descartado por año y la visibilidad de las películas
descartadas y conservadas.

In [7]:
display((df.groupby("anio_estreno")["descartada"].mean() * 100).round(1))

display(df.groupby("descartada").agg(
    peliculas=("id", "size"),
    pct_con_votos=("vote_count", lambda s: round((s > 0).mean() * 100, 1)),
    pct_con_poster=("poster_path", lambda s: round(s.notna().mean() * 100, 1)),
))

anio_estreno
2000    11.7
2001    13.0
2002    13.7
2003    16.1
2004    18.8
2005    19.5
2006    18.0
2007    17.4
2008    16.2
2009    16.0
2010    15.3
2011    14.8
2012    13.2
2013    13.8
2014    13.8
2015    12.3
2016    11.7
2017    11.4
2018    11.2
2019    10.5
2020    10.1
2021    10.2
2022     9.0
2023     8.6
2024     7.5
2025     7.4
Name: descartada, dtype: float64

,peliculas,pct_con_votos,pct_con_poster
descartada,,,
False,602451,41.5,84.7
True,79417,22.0,72.8


El porcentaje de películas descartadas varía entre años: es del 11,7 %
en 2000, alcanza un máximo del 19,5 % en 2005 y disminuye hasta el
7,4 % en 2025. Las películas descartadas tienen menos visibilidad que
las conservadas: el 22,0 % tiene votos frente al 41,5 %, y el 72,8 %
tiene póster frente al 84,7 %.

Por tanto, el descarte no es aleatorio. El análisis describe las
películas con dirección registrada en TMDB, no todo el cine del
periodo, y la evolución temporal debe interpretarse con cautela.

## 5. Guardado de los datos

Guardo dos versiones: la tabla completa con el grupo de dirección, y la
base del análisis, sin las películas descartadas.

In [8]:
df.to_pickle(carpeta_proc / "peliculas_todas_con_grupo.pkl")

df_analisis = df[~df["descartada"]].copy()
df_analisis.to_pickle(carpeta_proc / "peliculas_con_direccion.pkl")

tam = df_analisis["grupo_direccion"].value_counts()
display(pd.DataFrame({"n": tam, "%": (tam / len(df_analisis) * 100).round(2)}))

,n,%
grupo_direccion,,
No identificado,322770,53.58
Hombres,219028,36.36
Mujeres,56466,9.37
Mixto,4187,0.69


## Resultado

De las 681.868 películas, 602.451 tienen dirección acreditada y forman
la base del análisis. Las 79.417 restantes (79.385 sin director
acreditado y 32 sin créditos descargados) se excluyen del análisis de
dirección y se registran en `peliculas_sin_director.csv`.

Dentro de la base del análisis, el 53,6 % de las películas está en
«No identificado», porque alguna persona de la dirección no tiene el
género especificado en TMDB. Este es el principal límite del estudio,
y por eso los porcentajes por grupo se calcularán sobre las películas
de dirección conocida (Mujeres, Hombres y Mixto).

Archivos que genera este notebook, en `src/data/processed/`:

- `tramos_direccion/`: lectura intermedia de los créditos, por tramos.
- `direccion.pkl`: una fila por persona y película con el puesto `Director`.
- `ids_con_creditos.pkl`: identificadores de las películas con créditos.
- `peliculas_sin_director.csv`: películas descartadas y su motivo.
- `peliculas_todas_con_grupo.pkl`: todas las películas con su grupo.
- `peliculas_con_direccion.pkl`: base del análisis, sin descartadas.

El siguiente notebook, `03a_analisis_visualizacion`, analiza los géneros.